In [19]:
import os
import gc 
import numpy as np
import h5py
from tqdm import tqdm
import pandas as pd

sdss_reg_mu    = np.load("/home/arbiter/projects/Survey-invariant-generalization/src/models/reg_mu.npy")      # SDSS mean
sdss_reg_sigma = np.load("/home/arbiter/projects/Survey-invariant-generalization/src/models/reg_sigma.npy")   # SDSS std

def get_flux_labels(data, meta_data):
    
    flux_list = []
    teff_list = []
    logg_list = []
    feh_list  = []
    cls_list  = []
    failed    = 0

    class_map = {"O": 0, "B": 1, "A": 2, "F": 3, "G": 4, "K": 5, "M": 6}

    for star_id in tqdm(data):
        try:
            # Flux
            flux_array = data[star_id]["flux"][:]

            # Strip leading '1' to match obsid in metadata
            obsid = int(star_id)

            # Metadata lookup
            star_meta = meta_data.loc[meta_data["obsid"] == obsid]

            # Skip if no match found
            if len(star_meta) == 0:
                failed += 1
                continue

            teff = star_meta["teff"].iloc[0]
            logg = star_meta["logg"].iloc[0]
            feh  = star_meta["feh"].iloc[0]
            spectral_class = star_meta["subclass"].iloc[0][0]

            # Skip if any label is null
            if pd.isna(teff) or pd.isna(logg) or pd.isna(feh):
                failed += 1
                continue

            # Skip if class not in map
            if spectral_class not in class_map:
                failed += 1
                continue

            flux_list.append(flux_array)
            teff_list.append(teff)
            logg_list.append(logg)
            feh_list.append(feh)
            cls_list.append(class_map[spectral_class])

        except Exception as e:
            print(f"Failed for {star_id}: {e}")
            failed += 1
            continue

    print(f"Collected: {len(flux_list)} | Failed/skipped: {failed}")

    flux_array = np.array(flux_list)
    y_reg      = np.column_stack([teff_list, logg_list, feh_list])
    y_cls      = np.array(cls_list)

    return flux_array, y_reg, y_cls

def normalise_labels(y_reg):
    '''
    Standardise each parameter to zero mean unit variance.
    Returns normalised array + stats for denormalisation later.
    '''
    mu    = y_reg.mean(axis=0)    # shape -> (3,)
    sigma = y_reg.std(axis=0)     # shape -> (3,)
    return (y_reg - mu) / sigma, mu, sigma


def denormalise_labels(y_norm, mu, sigma):
    return y_norm * sigma + mu

def clean_stars(flux, y_reg, y_cls):

    per_star_median = np.median(flux, axis=1)
    good_norm_mask  = per_star_median < 5.0

    flux  = flux[good_norm_mask]
    y_reg = y_reg[good_norm_mask]
    y_cls = y_cls[good_norm_mask]

    print(f"After norm filter  — kept: {good_norm_mask.sum()} | dropped: {(~good_norm_mask).sum()}")

    star_index_list = []
    for i in range(len(flux)):

        star = flux[i]
        star = star[(star > 10) | (star < -3.0)]
        if len(star) > 0:
            star_index_list.append(i)

    bad_indices  = np.array(star_index_list)
    quality_mask = np.ones(len(flux), dtype=bool)
    quality_mask[bad_indices] = False

    flux_clean  = flux[quality_mask]
    y_reg_clean = y_reg[quality_mask]
    y_cls_clean = y_cls[quality_mask]

    return flux_clean, y_reg_clean, y_cls_clean    

data_path = "/home/arbiter/projects/Survey-invariant-generalization/data/usable_data"
lamost_data = h5py.File(f"{data_path}/lamost_resampled.h5")
lamost_meta_data = pd.read_csv(f"{data_path}/lamost_meta.csv")

flux, y_reg, y_cls = get_flux_labels(lamost_data, lamost_meta_data)
flux_clean, y_reg_clean, y_cls_clean = clean_stars(flux, y_reg, y_cls)

print(flux_clean.min())
print(flux_clean.max())

y_reg_norm = (y_reg_clean - sdss_reg_mu) / sdss_reg_sigma

np.save("lamost_y_reg_raw.npy", y_reg_clean)  # save raw physical values
np.save("lamost_y_cls.npy", y_cls_clean)      # for H1 UMAP coloring
np.save("lamost_flux.npy", flux_clean)         # might need later

100%|██████████| 20000/20000 [00:11<00:00, 1741.75it/s]


Collected: 20000 | Failed/skipped: 0
After norm filter  — kept: 20000 | dropped: 0
-2.9268791365345046
9.994687242589178


In [3]:
for i, star_id in enumerate(lamost_data):
    print(star_id)
    if i > 5:
        break

1001017
1001058
1001063
1001064
1001066
1001070
1001071


In [4]:
star_id = "1001017"
print(list(lamost_data[star_id].keys()))
print(lamost_data[star_id]["flux"][:5])  # first 5 flux values

['flux', 'wavelength_grid']
[0.         1.28473149 1.20266595 1.43321165 0.5239075 ]


In [2]:
print(lamost_meta_data["obsid"])
print(lamost_meta_data["obsid"].dtype)

0         101001
1         101005
2         101008
3         101009
4         101016
          ...   
19995    1603020
19996    1603021
19997    1603023
19998    1603025
19999    1603026
Name: obsid, Length: 20000, dtype: int64
int64


In [ ]:
for star_id in lamost_data:
    print(star_id)

In [6]:
lamost_data["1001017"].keys()
lamost_meta_data.head()

,Unnamed: 0,obsid,uid,obsdate,lmjd,mjd,planid,spid,fiberid,class,subclass,z,ra,dec,rv,logg,teff,feh
0,0,101001,G12763466510210,2011-10-24,55859,55858,F5902,1,1,STAR,G7,-0.000139,332.202274,-2.056767,-41.58,4.545,5178.32,-0.497
1,1,101005,G12763553667654,2011-10-24,55859,55858,F5902,1,5,STAR,G0,-0.000251,332.535456,-2.116436,-75.14,3.722,5314.64,-0.737
2,2,101008,G12763564851128,2011-10-24,55859,55858,F5902,1,8,STAR,F9,0.000087,332.368745,-1.955771,26.20,4.594,5580.10,-0.019
3,3,101009,G12763486062568,2011-10-24,55859,55858,F5902,1,9,STAR,G0,-0.000115,332.206665,-1.868653,-34.54,4.090,5555.26,-0.470
4,4,101016,G12763535296239,2011-10-24,55859,55858,F5902,1,16,STAR,K3,-0.000044,332.348725,-2.136096,-13.11,4.691,4588.43,-0.228


In [17]:
for star_id in lamost_data:
    obsid = int(star_id)
    print(obsid)

1001017
1001058
1001063
1001064
1001066
1001070
1001071
1001072
1001106
1001117
1001124
1001127
1001134
1001135
1001139
1001143
1001144
1001146
1001148
1001152
1001153
1001168
1001172
1001173
1001178
1001188
1001202
1001220
1002011
1002028
1002029
1002030
1002031
1002036
1002040
1002043
1002050
1002054
1002057
1002060
1002063
1002067
1002072
1002074
1002079
1002093
1002094
1002096
1002097
1002122
1002126
1002129
1002133
1002142
1002152
1002155
1002156
1002160
1002165
1002179
1002180
1002191
1002204
1002239
1002240
1002243
1003004
1003007
1003010
1003027
1003029
1003033
1003034
1003037
1003040
1003047
1003048
1003049
1003053
1003058
1003062
1003066
1003073
1003074
1003075
1003077
1003079
1003086
1003087
1003102
1003103
1003106
1003108
1003109
1003115
1003122
1003127
1003128
1003134
1003143
1003144
1003146
1003147
1003153
1003163
1003171
1003172
1003182
1003196
1003198
1003200
1003201
1003204
1003205
1003206
1003207
1003208
1003210
1003211
1003216
1003217
1003219
1003221
1003227
1003232


In [15]:
counter = 0

for i in lamost_meta_data["obsid"]:
    for j in lamost_data:
        if j == str(i):
            counter+=1 

print(counter)

KeyboardInterrupt: 